# Chapter 5 — Claude Code: Configuration and Workflows · Practice

**Goal:** build a throwaway Claude Code project from Python, predict exactly what loads into Claude's context, and then drive Claude Code headless the way a CI job does.

- **Theory:** [README.md](README.md). This notebook holds the code; the explanations live in the README.
- **Estimated time:** 3.5–4.5 hours
- **Estimated API cost:** **$0** with the default `RUN_CLAUDE = False`, because every live cell falls back to mock output. With `RUN_CLAUDE = True`, the dozen or so live calls cost roughly $0.60–$1.80 with the default model. Each call is capped with `--max-budget-usd`.
- **Prerequisites:** the repo venv from [00 — Prerequisites](../../00-prerequisites/README.md). For the live cells you also need Claude Code installed and signed in (`claude --version` works), or an `ANTHROPIC_API_KEY` in `.env`.

**How this notebook works**

- Everything happens in a **sandbox project in a temp directory**. Your real `~/.claude` and this repo are never written to.
- Live cells call `claude -p ... --output-format json` through `subprocess`. They only run when you set `RUN_CLAUDE = True` **and** `claude` is on your `PATH`. Otherwise they print the command and use a realistic mock result, so you can learn the parsing code without credit.
- Interactive-only features (`Shift+Tab`, `/memory`, `/compact`, `/context`) are covered as short terminal checklists.

> **Safety note:** `claude -p` skips the workspace trust dialog and loads the folder's hooks, `.mcp.json` servers and CLAUDE.md without asking. Only run it in directories you trust. The sandbox below is one you just created yourself.

## Setup

This chapter drives **Claude Code (the CLI)**, not the Messages API, so there is no `anthropic.Anthropic()` client here. `MODEL` is passed to the CLI with `--model`.

In [ ]:
import difflib
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import tempfile
import textwrap
from pathlib import Path
from typing import Any

import yaml  # PyYAML ships with JupyterLab; used to parse frontmatter the way Claude Code does
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
RUN_CLAUDE = False           # set to True to make real `claude -p` calls (costs money)
CLAUDE_BIN = shutil.which("claude")
MAX_BUDGET_USD = "0.25"      # cap for every live call (--max-budget-usd)

print(f"model={MODEL}")
print(f"claude CLI: {CLAUDE_BIN or 'not on PATH'}   RUN_CLAUDE={RUN_CLAUDE}")

In [ ]:
# Live prerequisites. They are only enforced when you opt in to live calls,
# so the offline cells keep working if you skip this cell.
if RUN_CLAUDE:
    assert CLAUDE_BIN, "Install Claude Code and put `claude` on PATH (see 00-prerequisites)"
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("No ANTHROPIC_API_KEY in .env: live cells use your Claude Code login, and --bare runs are skipped.")
    print("Live mode: real `claude -p` calls will run.")
else:
    print("Offline mode: live cells print the command they would run and use mock output.")

### Sandbox and helpers

`SANDBOX` is a fresh temp directory. Inside it, `acme-shop/` plays the repo you work in, and `fake-home/` stands in for `~` **in the offline model only**. A real `claude` run still reads your real `~/.claude/CLAUDE.md` (user scope), which is exactly how the hierarchy works.

In [ ]:
SANDBOX = Path(tempfile.mkdtemp(prefix="cc-ch5-")).resolve()
PROJECT = SANDBOX / "acme-shop"       # the "repo"; live runs use it as the working directory
FAKE_HOME = SANDBOX / "fake-home"     # stands in for ~ in the offline model


def write(path: Path, text: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"))
    return path


def print_tree(root: Path, prefix: str = "") -> None:
    if not prefix:
        print(root.name + "/")
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(f"{prefix}{'└── ' if last else '├── '}{p.name}{'/' if p.is_dir() else ''}")
        if p.is_dir():
            print_tree(p, prefix + ("    " if last else "│   "))


def markers_in(text: str) -> set[str]:
    """Every file in the sandbox carries a unique `MARKER: NAME` line, so we can tell what Claude saw."""
    return set(re.findall(r"MARKER: ([A-Z0-9-]+)", text or ""))


print("sandbox:", SANDBOX)

In [ ]:
def mock_result(result: str = "", structured: Any = None, session_id: str = "00000000-mock-0000", **extra) -> dict:
    """Shape of `claude -p --output-format json` output (one JSON object on stdout)."""
    out = {
        "type": "result", "subtype": "success", "is_error": False,
        "duration_ms": 4210, "duration_api_ms": 3900, "num_turns": 2,
        "result": result, "session_id": session_id, "total_cost_usd": 0.0123,
        "usage": {"input_tokens": 5120, "output_tokens": 310, "cache_read_input_tokens": 4096},
        "permission_denials": [],
    }
    if structured is not None:
        out["structured_output"] = structured
    out.update(extra)
    return out


def run_claude(argv: list[str], cwd: Path, stdin: str | None = None,
               mock: dict | None = None, timeout: int = 600) -> dict:
    """Run a headless Claude Code command and return its parsed JSON result.

    With RUN_CLAUDE = False nothing is executed: the command is printed and `mock` is returned.
    """
    print(f"$ cd {cwd.name} && {shlex.join(argv)[:400]}")
    if not (RUN_CLAUDE and CLAUDE_BIN):
        print("  [offline] not executed; using mock output\n")
        return mock or {}
    proc = subprocess.run(argv, cwd=cwd, input=stdin, capture_output=True, text=True,
                          timeout=timeout, env=os.environ.copy())
    # Do not check the exit code first: a run that fails *inside* (max turns, budget, auth)
    # still prints its JSON result on stdout. Only flag errors reach stderr.
    if not proc.stdout.strip():
        raise RuntimeError(f"claude exited {proc.returncode} without JSON: {proc.stderr[:500]}")
    return json.loads(proc.stdout)


def show(out: dict) -> None:
    keys = ("subtype", "is_error", "num_turns", "session_id", "total_cost_usd")
    print({k: out.get(k) for k in keys})
    print("result:", (out.get("result") or "")[:800])

## 5.1 The CLAUDE.md hierarchy

CLAUDE.md files at several scopes (managed, user, project, local, subdirectory) are **concatenated, not overridden**. Files in the working directory and every directory above it load at launch; subdirectory files load lazily, when Claude first reads a file there. CLAUDE.md is context, not enforcement. [Theory](README.md#51-the-claudemd-hierarchy)

First, build the memory files at every level we can fake in a sandbox. There is no managed-policy file here, because that path needs admin rights.

In [ ]:
write(SANDBOX / "CLAUDE.md", """
    # Monorepo-wide notes (an ancestor of the project directory)
    MARKER: ANCESTOR
    - All services deploy through the shared pipeline; never hand-edit CI secrets.
    """)
write(FAKE_HOME / ".claude" / "CLAUDE.md", """
    # My personal preferences (user scope, used by the offline model only)
    MARKER: USER
    - Prefer short answers.
    """)
write(PROJECT / "CLAUDE.md", """
    # acme-shop
    MARKER: PROJECT
    Build with `make build`; run tests with `make test`.

    - Coding standards: @docs/standards.md
    - Background reading (mentioned, NOT imported): `@docs/not-imported.md`
    """)
write(PROJECT / "CLAUDE.local.md", """
    MARKER: LOCAL
    - My sandbox API runs on http://localhost:8081 (personal, never committed).
    """)
write(PROJECT / ".gitignore", "CLAUDE.local.md\n")
write(PROJECT / "packages" / "api" / "CLAUDE.md", """
    MARKER: SUBDIR-API
    - Handlers return {"error": {"code": ..., "message": ...}} on failure.
    """)

# Some source files so there is something to read
write(PROJECT / "packages" / "api" / "handlers.py", """
    def get_price(sku: str) -> dict:
        return {"sku": sku, "price": 100.0}
    """)
write(PROJECT / "src" / "pricing.py", """
    def apply_discount(price: float, code: str) -> float:
        if code == "SAVE10":
            return round(price * 0.9, 2)
        return price
    """)
write(PROJECT / "tests" / "test_pricing.py", """
    from src.pricing import apply_discount


    def test_save10():
        assert apply_discount(100, "SAVE10") == 90
    """)
write(PROJECT / "services" / "billing" / "db" / "migrations" / "001_init.sql",
      "CREATE TABLE invoices (id serial primary key);\n")

print_tree(SANDBOX)

Now a **simplified model** of the documented loading rules. It is not Claude Code's implementation, just enough to make predictions we can check against a real run:

- at launch: user file, then `CLAUDE.md` / `.claude/CLAUDE.md` / `CLAUDE.local.md` from the filesystem root down to the working directory (local last at each level), plus unscoped `.claude/rules/*.md`
- on demand: a subdirectory's CLAUDE.md, when Claude reads a file in it

In [ ]:
def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Split `---` YAML frontmatter from the body. Unparsable YAML -> treated as no frontmatter."""
    m = re.match(r"\A---[ \t]*\n(.*?)\n---[ \t]*(?:\n|\Z)", text, flags=re.S)
    if not m:
        return {}, text
    try:
        meta = yaml.safe_load(m.group(1)) or {}
    except yaml.YAMLError:
        return {}, text
    return (meta if isinstance(meta, dict) else {}), text[m.end():]


def rules_in(project: Path, scoped: bool) -> list[Path]:
    rules = sorted((project / ".claude" / "rules").glob("**/*.md"))
    return [r for r in rules if bool(parse_frontmatter(r.read_text())[0].get("paths")) == scoped]


def launch_files(cwd: Path, home: Path = FAKE_HOME) -> list[tuple[str, Path]]:
    """What loads at launch, broadest scope first (our walk stops at SANDBOX; the real CLI goes up to /)."""
    found: list[tuple[str, Path]] = []
    user = home / ".claude" / "CLAUDE.md"
    if user.exists():
        found.append(("user", user))
    found += [("user rule", r) for r in sorted((home / ".claude" / "rules").glob("**/*.md"))]
    chain = [d for d in [cwd, *cwd.parents] if d.is_relative_to(SANDBOX)][::-1]
    for d in chain:
        scope = "project" if d == cwd else "ancestor"
        for name in ("CLAUDE.md", ".claude/CLAUDE.md"):
            if (d / name).exists():
                found.append((scope, d / name))
        if d == cwd:
            found += [("rule (unscoped)", r) for r in rules_in(d, scoped=False)]
        if (d / "CLAUDE.local.md").exists():
            found.append(("local", d / "CLAUDE.local.md"))
    return found


for scope, path in launch_files(PROJECT):
    print(f"{scope:16} {path.relative_to(SANDBOX)}")

> **Observe:** the ancestor file (`SANDBOX/CLAUDE.md`) loads even though it is outside the project, because it is *above* the working directory. `packages/api/CLAUDE.md` is missing from the launch list: it is below the working directory, so it waits until Claude reads a file in `packages/api/`. `CLAUDE.local.md` comes last at its level, so your personal notes are read after the team's.

> **Try it:** call `launch_files(PROJECT / "packages" / "api")`. Starting Claude *inside* the subdirectory makes the subdirectory file part of the launch set, and `acme-shop/CLAUDE.md` becomes an ancestor.

## 5.2 `@path` imports: modular CLAUDE.md

`@path` pulls another file into context **at launch**, next to the file that imports it. Relative paths resolve from the importing file, imports can nest **up to four hops**, and text inside backticks is not imported. Imports organize; they do not save tokens. [Theory](README.md#52-path-imports-modular-claudemd)

This section builds the planned experiment: a **five-hop import chain**. The model predicts that hop 5 is cut off.

In [ ]:
write(PROJECT / "docs" / "standards.md", """
    MARKER: IMPORT-HOP-1
    - Type-annotate every public function.
    - Deeper rules: @hops/hop2.md
    """)
for n in range(2, 5):
    write(PROJECT / "docs" / "hops" / f"hop{n}.md", f"MARKER: IMPORT-HOP-{n}\n- Next level: @hop{n + 1}.md\n")
write(PROJECT / "docs" / "hops" / "hop5.md", "MARKER: IMPORT-HOP-5\n- One hop too deep.\n")
write(PROJECT / "docs" / "not-imported.md", "MARKER: CODE-SPAN\n- Mentioned in backticks, so never loaded.\n")

IMPORT_RE = re.compile(r"(?<![\w`])@((?:[^\s`\\]|\\ )+)")
MAX_HOPS = 4


def find_imports(markdown: str) -> list[str]:
    """`@path` references outside fenced blocks and code spans (a backslash escapes a space)."""
    without_code = re.sub(r"`[^`]*`", "", re.sub(r"```.*?```", "", markdown, flags=re.S))
    return [m.group(1).replace("\\ ", " ").rstrip(".,;:)") for m in IMPORT_RE.finditer(without_code)]


def expand(file: Path, hop: int = 0, seen: set | None = None) -> list[tuple[int, Path]]:
    seen = set() if seen is None else seen
    if file in seen or not file.exists():
        return []
    seen.add(file)
    loaded = [(hop, file)]
    if hop == MAX_HOPS:          # an import inside this file would be hop 5
        return loaded
    for ref in find_imports(file.read_text()):
        target = FAKE_HOME / ref[2:] if ref.startswith("~/") else file.parent / ref
        loaded += expand(target.resolve(), hop + 1, seen)
    return loaded


for hop, path in expand(PROJECT / "CLAUDE.md"):
    print("  " * hop + f"hop {hop}: {path.relative_to(PROJECT)}")

**Imports do not reduce context cost.** Compare a monolithic CLAUDE.md with the same content split into imports (≈4 characters per token is a rough rule of thumb):

In [ ]:
def approx_tokens(paths: list[Path]) -> int:
    return sum(len(p.read_text()) for p in paths) // 4


expanded = [p for _, p in expand(PROJECT / "CLAUDE.md")]
monolith = write(SANDBOX / "monolith" / "CLAUDE.md", "".join(p.read_text() for p in expanded))
print(f"split into {len(expanded)} files via @imports: ~{approx_tokens(expanded)} tokens at launch")
print(f"one monolithic file:                   ~{approx_tokens([monolith])} tokens at launch")

### Experiment: what does Claude actually see at launch?

Predict the markers from the model, then ask a real headless run to list the `MARKER:` lines in its context. `--tools ""` removes every built-in tool, so Claude cannot read files to cheat. Only launch-time context is visible.

In [ ]:
predicted_launch = set()
for scope, path in launch_files(PROJECT):
    files = [p for _, p in expand(path)] if path.name in ("CLAUDE.md", "CLAUDE.local.md") else [path]
    predicted_launch |= set().union(*(markers_in(p.read_text()) for p in files))
predicted_launch.discard("USER")      # the real run reads your real ~/.claude, not FAKE_HOME

MARKER_PROMPT = ("List every line that starts with 'MARKER:' that appears in your instructions, "
                 "memory files or rules. Output only those lines, one per line.")
argv = ["claude", "-p", MARKER_PROMPT, "--output-format", "json", "--model", MODEL,
        "--tools", "", "--max-turns", "2", "--max-budget-usd", MAX_BUDGET_USD,
        "--no-session-persistence"]
out = run_claude(argv, cwd=PROJECT,
                 mock=mock_result("\n".join(f"MARKER: {m}" for m in sorted(predicted_launch))))
seen = markers_in(out.get("result", ""))
print("predicted:", sorted(predicted_launch))
print("seen:     ", sorted(seen))
print("missing from run:", sorted(predicted_launch - seen) or "-", "| unexpected:", sorted(seen - predicted_launch) or "-")

> **Observe:** `IMPORT-HOP-5`, `CODE-SPAN` and `SUBDIR-API` should be absent: the first is past the four-hop limit, the second sits inside backticks, and the third is lazy. If a live run disagrees with the model, trust the run and check `/context` in an interactive session. The docs describe behavior, and versions change.

> **Try it:** add `@hops/hop2.md` directly to `CLAUDE.md`. That shortens the chain, and `IMPORT-HOP-5` now sits at hop 4. Re-run the prediction cell.

## 5.3 `.claude/rules/`: topic files and path-scoped rules

A rule without frontmatter loads at launch, like `.claude/CLAUDE.md`. A rule with `paths:` globs loads only when Claude **reads, writes or edits** a matching file. `paths` accepts a YAML list or a comma-separated string, and brace expansion works. If the frontmatter YAML does not parse, the rule loads as if it had no `paths`. [Theory](README.md#53-clauderules-topic-files-and-path-scoped-rules)

In [ ]:
write(PROJECT / ".claude" / "rules" / "style.md", """
    # Style (unscoped: loads every session)
    MARKER: RULE-UNSCOPED
    - Use f-strings, never %-formatting.
    """)
write(PROJECT / ".claude" / "rules" / "testing.md", """
    ---
    paths:
      - "**/test_*.py"
      - "**/*_test.py"
    ---
    MARKER: RULE-TESTS
    - Use pytest fixtures; unit tests never touch the network.
    """)
write(PROJECT / ".claude" / "rules" / "migrations.md", """
    ---
    paths: "services/*/db/migrations/**"
    ---
    MARKER: RULE-MIGRATIONS
    - Every migration needs a reversible down step.
    """)
write(PROJECT / ".claude" / "rules" / "frontend" / "react.md", """
    ---
    paths: [src/**/*.{ts,tsx}
    ---
    MARKER: RULE-BROKEN-YAML
    - The flow list above is never closed, so this YAML does not parse.
    """)

for r in sorted((PROJECT / ".claude" / "rules").glob("**/*.md")):
    meta, _ = parse_frontmatter(r.read_text())
    print(f"{str(r.relative_to(PROJECT)):40} paths={meta.get('paths', '— (loads at launch)')}")

In [ ]:
def expand_braces(pattern: str) -> list[str]:
    m = re.search(r"\{([^{}]*)\}", pattern)
    if not m:
        return [pattern]
    return [x for alt in m.group(1).split(",")
            for x in expand_braces(pattern[:m.start()] + alt + pattern[m.end():])]


def glob_to_regex(pattern: str) -> re.Pattern:
    out, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            out, i = out + "(?:.*/)?", i + 3     # zero or more directories
        elif pattern.startswith("**", i):
            out, i = out + ".*", i + 2
        elif pattern[i] == "*":
            out, i = out + "[^/]*", i + 1        # never crosses a /
        elif pattern[i] == "?":
            out, i = out + "[^/]", i + 1
        else:
            out, i = out + re.escape(pattern[i]), i + 1
    return re.compile(out + r"\Z")


def normalize_paths(value) -> list[str]:
    """YAML list, or one comma-separated string (commas inside {a,b} braces do not split)."""
    items = value if isinstance(value, list) else re.split(r",(?![^{]*\})", str(value or ""))
    return [str(s).strip() for s in items if str(s).strip()]


def rule_matches(paths_value, rel_path: str) -> bool:
    return any(glob_to_regex(p).match(rel_path)
               for g in normalize_paths(paths_value) for p in expand_braces(g))


def on_read(file: Path, cwd: Path = PROJECT) -> list[tuple[str, Path]]:
    """What a Read/Write/Edit of `file` pulls in lazily (subdirectory CLAUDE.md + matching path rules)."""
    extra = []
    for d in [d for d in [file.parent, *file.parent.parents] if d.is_relative_to(cwd) and d != cwd][::-1]:
        for name in ("CLAUDE.md", "CLAUDE.local.md"):
            if (d / name).exists():
                extra.append(("subdir", d / name))
    rel = file.relative_to(cwd).as_posix()
    for rule in rules_in(cwd, scoped=True):
        if rule_matches(parse_frontmatter(rule.read_text())[0]["paths"], rel):
            extra.append(("path rule", rule))
    return extra


targets = ["src/pricing.py", "tests/test_pricing.py", "packages/api/handlers.py",
           "services/billing/db/migrations/001_init.sql"]
for t in targets:
    loaded = [p.relative_to(PROJECT).as_posix() for _, p in on_read(PROJECT / t)]
    print(f"read {t:45} -> {loaded or 'nothing extra'}")

> **Observe:** `frontend/react.md` has broken YAML, so it behaves as an **unscoped** rule and costs context in every session. `claude --debug` shows the parse error. One `.claude/rules/migrations.md` covers every service's migrations; you would need a CLAUDE.md in each `services/*/db/migrations/` folder to get the same effect without rules.

### Experiment: rules and subdirectory files load on read

Same marker question as before, but this time Claude may use only `Read` and must first read two files.

In [ ]:
predicted_after_read = set(predicted_launch)
for t in ("tests/test_pricing.py", "packages/api/handlers.py"):
    for _, p in on_read(PROJECT / t):
        predicted_after_read |= markers_in(p.read_text())
predicted_after_read |= set().union(*(markers_in(r.read_text()) for r in rules_in(PROJECT, scoped=False)))

prompt = ("Read tests/test_pricing.py and packages/api/handlers.py. Then " + MARKER_PROMPT[0].lower()
          + MARKER_PROMPT[1:])
argv = ["claude", "-p", prompt, "--output-format", "json", "--model", MODEL,
        "--tools", "Read", "--allowedTools", "Read", "--max-turns", "5",
        "--max-budget-usd", MAX_BUDGET_USD, "--no-session-persistence"]
out = run_claude(argv, cwd=PROJECT,
                 mock=mock_result("\n".join(f"MARKER: {m}" for m in sorted(predicted_after_read))))
seen = markers_in(out.get("result", ""))
print("new since the 5.2 run (predicted):", sorted(predicted_after_read - predicted_launch))
print("new since the 5.2 run (seen):     ", sorted(seen - predicted_launch))
print("RULE-MIGRATIONS loaded?", "RULE-MIGRATIONS" in seen, "(expected: False)")

> **Try it:** change the prompt to read `services/billing/db/migrations/001_init.sql` instead and predict which marker appears. Then turn `migrations.md` into an unscoped rule (delete the frontmatter) and note that it now costs context in every session, including sessions that never touch SQL.

## 5.4 Custom slash commands → skills

Custom commands were merged into skills: `.claude/commands/review.md` and `.claude/skills/review/SKILL.md` both create `/review`. When both exist, the **skill wins**. Across scopes, enterprise beats personal, and personal beats project; plugin skills are namespaced (`/plugin:name`) so they never clash. [Theory](README.md#54-custom-slash-commands--skills)

In [ ]:
write(PROJECT / ".claude" / "commands" / "review.md", """
    ---
    description: Legacy review command (old flat-file format)
    ---
    MARKER: LEGACY-COMMAND
    Review the staged changes for bugs.
    """)
write(PROJECT / ".claude" / "skills" / "review" / "SKILL.md", """
    ---
    name: review
    description: Review code changes for correctness bugs and missing tests. Use when the user asks to review a diff, a branch or a file.
    argument-hint: "[path-or-git-ref]"
    ---
    MARKER: TEAM-REVIEW-SKILL
    Review $ARGUMENTS for correctness bugs. Report file:line, severity and a suggested fix.
    """)
# A personal copy with the same name shadows the team skill for you only
write(FAKE_HOME / ".claude" / "skills" / "review" / "SKILL.md", """
    ---
    name: review
    description: Review code changes for correctness bugs. Use when the user asks for a review.
    ---
    MARKER: MY-REVIEW-SKILL
    Same as the team review, but also flag TODO comments.
    """)

SCOPE_ORDER = ["enterprise", "personal", "project"]   # highest precedence first


def discover(project: Path, home: Path) -> dict[str, list[tuple[str, str, Path]]]:
    found: dict[str, list[tuple[str, str, Path]]] = {}
    for scope, base in (("personal", home / ".claude"), ("project", project / ".claude")):
        for p in sorted((base / "skills").glob("*/SKILL.md")):
            name = parse_frontmatter(p.read_text())[0].get("name") or p.parent.name
            found.setdefault(name, []).append((scope, "skill", p))
        for p in sorted((base / "commands").glob("*.md")):
            found.setdefault(p.stem, []).append((scope, "command", p))
    return found


def winner(entries: list[tuple[str, str, Path]]) -> tuple[str, str, Path]:
    # Scope decides first; within one scope a skill beats a same-named command.
    return min(entries, key=lambda e: (SCOPE_ORDER.index(e[0]), e[1] != "skill"))


for name, entries in discover(PROJECT, FAKE_HOME).items():
    scope, kind, path = winner(entries)
    print(f"/{name}: candidates={[(s, k) for s, k, _ in entries]} -> {scope} {kind} {sorted(markers_in(path.read_text()))}")

> **Observe:** for *you*, `/review` resolves to the personal skill (`MY-REVIEW-SKILL`). A teammate without that file gets the project skill, and nobody gets the legacy command any more. Personal shadowing is the clean way to tweak a team skill without touching the team's version. A differently named copy (`/my-review`) also works, but two near-identical descriptions confuse automatic skill selection.

> **Try it:** delete `FAKE_HOME/.claude/skills/review/` and re-run the cell. Then delete the project skill too, and the legacy command becomes the winner.

## 5.5 Skills in depth: SKILL.md frontmatter

A skill is a directory. Claude sees each skill's `description` up front and loads the body only when the skill is invoked. Key fields: `disable-model-invocation` (only humans may trigger it), `context: fork` + `agent` (isolated subagent), `allowed-tools` (**pre-approves**, does not restrict), `disallowed-tools` (actually removes tools), and `` !`cmd` `` (shell output injected before Claude sees the skill). [Theory](README.md#55-skills-in-depth-skillmd-frontmatter)

In [ ]:
write(PROJECT / "CHANGES.txt", """
    feat: SAVE50 discount code (#412)
    fix: rounding error on totals above 10k (#415)
    chore: bump pytest (#416)
    """)
write(PROJECT / ".claude" / "skills" / "release-notes" / "SKILL.md", """
    ---
    name: release-notes
    description: Draft customer-facing release notes from CHANGES.txt. Use when the user asks for release notes, a changelog entry or a summary of a version.
    argument-hint: "[version]"
    disable-model-invocation: true
    allowed-tools: Bash(cat CHANGES.txt)
    disallowed-tools: WebFetch WebSearch
    ---
    Write release notes for version $ARGUMENTS.

    ## Raw changes
    !`cat CHANGES.txt`

    Group the changes under "Added" and "Fixed". Skip chores. At most 5 bullets. No ticket numbers.
    """)
write(PROJECT / ".claude" / "skills" / "map-callers" / "SKILL.md", """
    ---
    name: map-callers
    description: Find every call site of a function across the repo and return a compact table. Use when the user asks who calls a function or where it is used.
    argument-hint: "[function-name]"
    context: fork
    agent: Explore
    ---
    Find every call site of `$ARGUMENTS`. Return only a markdown table: file | line | calling function.
    """)


def split_tools(value) -> list[str]:
    """allowed-tools / disallowed-tools: YAML list, or a space/comma-separated string. Keeps `Bash(git diff *)` whole."""
    if isinstance(value, list):
        return [str(v).strip() for v in value]
    return re.findall(r"[^\s,()]+(?:\([^)]*\))?", str(value or ""))


def lint_skill(path: Path) -> list[str]:
    meta, body = parse_frontmatter(path.read_text())
    issues = []
    name = str(meta.get("name") or path.parent.name)
    if not re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", name) or len(name) > 64:
        issues.append(f"name {name!r}: use lowercase letters, digits and hyphens (max 64)")
    desc = str(meta.get("description") or "")
    if len(desc) < 40 or "use when" not in desc.lower():
        issues.append("description is vague: say what it does AND when to use it")
    if len(desc + str(meta.get("when_to_use") or "")) > 1536:
        issues.append("description + when_to_use is over 1,536 chars and gets truncated in the listing")
    if meta.get("agent") and meta.get("context") != "fork":
        issues.append("`agent` only applies together with `context: fork`")
    if "!`" in body and not meta.get("allowed-tools"):
        issues.append("!`cmd` without allowed-tools: the invocation aborts if the command is not already allowed")
    return issues


for skill in sorted((PROJECT / ".claude" / "skills").glob("*/SKILL.md")):
    meta, _ = parse_frontmatter(skill.read_text())
    print(f"/{meta.get('name')}: model-invocable={not meta.get('disable-model-invocation', False)}, "
          f"fork={meta.get('context') == 'fork'}, agent={meta.get('agent', '-')}, "
          f"pre-approved={split_tools(meta.get('allowed-tools'))}, removed={split_tools(meta.get('disallowed-tools'))}")
    print("   lint:", lint_skill(skill) or "ok")

What Claude receives when you type `/release-notes 1.4.0`: arguments are substituted and `` !`cmd` `` placeholders are replaced by command output **before** the model sees anything. The renderer below emulates only `cat`, which is enough for this skill.

In [ ]:
def render_skill(path: Path, args: str, cwd: Path = PROJECT) -> str:
    _, body = parse_frontmatter(path.read_text())
    words = shlex.split(args)
    body = body.replace("$ARGUMENTS", args)
    body = re.sub(r"\$(\d+)", lambda m: words[int(m.group(1))] if int(m.group(1)) < len(words) else "", body)

    def run(m: re.Match) -> str:
        cmd = shlex.split(m.group(1))
        if cmd[0] == "cat" and len(cmd) == 2:
            return (cwd / cmd[1]).read_text().strip()
        return f"[would run: {m.group(1)}]"

    return re.sub(r"(?<!\S)!`([^`]+)`", run, body)


print(render_skill(PROJECT / ".claude" / "skills" / "release-notes" / "SKILL.md", "1.4.0"))

In [ ]:
# Live: user-invoked skills work in -p mode. disable-model-invocation only stops *Claude* from invoking it.
argv = ["claude", "-p", "/release-notes 1.4.0", "--output-format", "json", "--model", MODEL,
        "--max-turns", "3", "--max-budget-usd", MAX_BUDGET_USD, "--no-session-persistence"]
out = run_claude(argv, cwd=PROJECT, mock=mock_result(
    "## 1.4.0\n### Added\n- SAVE50 discount code\n### Fixed\n- Rounding error on totals above 10k"))
show(out)

> **Observe:** the skill's shell command ran *before* the model call, and the chore and ticket numbers are gone. `map-callers` runs as a forked **Explore** subagent, so its file-by-file search stays out of your main context and only the table comes back.

> **Try it:** remove `allowed-tools` from `release-notes` and run it live again. Depending on your permission settings, the `` !`cat` `` injection may now abort the invocation instead of prompting. A skill never prompts during rendering.

## 5.6 Plan mode vs direct execution

Plan mode is a permission mode: Claude reads files and may run exploratory commands, then writes a plan, but it **cannot edit source until you approve**. Use it for multi-file, ambiguous or architectural work; use direct execution for clear, local fixes. [Theory](README.md#56-plan-mode-vs-direct-execution)

In [ ]:
def choose_mode(files_touched: int, valid_designs: int, ambiguous: bool, familiar: bool) -> tuple[str, list[str]]:
    reasons = []
    if files_touched > 5:
        reasons.append(f"{files_touched} files")
    if valid_designs > 1:
        reasons.append(f"{valid_designs} viable designs")
    if ambiguous:
        reasons.append("ambiguous requirements")
    if not familiar:
        reasons.append("unfamiliar code")
    return ("plan", reasons) if reasons else ("direct", ["clear, local change"])


SCENARIOS = {
    "Fix NPE from a clear stack trace in one file": dict(files_touched=1, valid_designs=1, ambiguous=False, familiar=True),
    "Migrate logging library across 45 files":       dict(files_touched=45, valid_designs=2, ambiguous=False, familiar=True),
    "'Add Slack support' (webhook vs bot vs app)":    dict(files_touched=6, valid_designs=3, ambiguous=True, familiar=True),
    "Add one validation check you already designed": dict(files_touched=1, valid_designs=1, ambiguous=False, familiar=True),
}
for task, facts in SCENARIOS.items():
    mode, why = choose_mode(**facts)
    print(f"{mode:6} <- {task}  ({', '.join(why)})")

### Experiment: plan mode writes nothing

Hash every source file, run a refactor request in `--permission-mode plan`, and hash again. In `-p` mode there is no human to approve the plan, so the run ends with the plan as its result.

In [ ]:
def snapshot(root: Path) -> dict[str, str]:
    return {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(root.rglob("*")) if p.is_file()}


before = snapshot(PROJECT)
argv = ["claude", "-p", "Refactor src/pricing.py so discount codes come from a lookup table, and update the tests.",
        "--permission-mode", "plan", "--output-format", "json", "--model", MODEL,
        "--max-turns", "6", "--max-budget-usd", MAX_BUDGET_USD, "--no-session-persistence"]
out = run_claude(argv, cwd=PROJECT, mock=mock_result(
    "Plan:\n1. Add DISCOUNTS = {'SAVE10': 0.10} in src/pricing.py\n2. Look up the rate; unknown codes return the price\n"
    "3. Parametrize tests/test_pricing.py over the table"))
show(out)
after = snapshot(PROJECT)
changed = {k for k in before.keys() | after.keys() if before.get(k) != after.get(k)}
print("files changed by the plan-mode run:", sorted(changed) or "none")

> **Observe:** no file hashes changed. For discovery-heavy work, such as finding every call site in 120 files, pair planning with the read-only **Explore** subagent (or a `context: fork` skill like `map-callers`), so the verbose search stays out of your main context.

> **Try it:** in an interactive terminal inside the sandbox (`cd` to the path printed by `print(PROJECT)`), press `Shift+Tab` until the mode shows plan, or prefix a prompt with `/plan`. When the plan appears, press `Ctrl+G` to edit it before approving.

## 5.7 `/compact` and managing the context window

`/compact [focus]` replaces history with a structured summary. Project-root CLAUDE.md, unscoped rules, auto memory, the plan and invoked skill bodies are re-injected from disk. Path-scoped rules, nested CLAUDE.md files and anything you only typed in chat survive **only as far as the summary kept them**. Exact numbers and IDs go first. [Theory](README.md#57-compact-and-managing-the-context-window)

In [ ]:
AFTER_COMPACT = {
    "system prompt / output style":           "still applies",
    "project-root CLAUDE.md + unscoped rules": "re-injected from disk",
    "auto memory (MEMORY.md)":                 "re-injected from disk",
    "plan written in plan mode":               "re-injected from disk",
    "invoked skill bodies":                    "re-injected (5,000 tokens each, 25,000 total)",
    "skill listing (descriptions)":            "NOT re-injected",
    "path-scoped rules / nested CLAUDE.md":    "summarized away until a matching file is read again",
    "files read or edited":                    "up to 5 re-read, most recently modified first",
    "instructions typed in chat":              "only what the summary kept",
}
for item, fate in AFTER_COMPACT.items():
    print(f"{'!!' if fate.startswith(('summarized', 'only', 'NOT')) else '  '} {item:42} {fate}")

# Persist exact facts BEFORE compacting, then compact with a focus.
facts = {"error_code": "ERR-4471", "failing_test": "tests/test_pricing.py::test_save10", "repro_sku": "SKU-90210"}
notes = write(PROJECT / "NOTES.md", "# Session notes (survive /compact)\n" +
              "\n".join(f"- {k}: {v}" for k, v in facts.items()) + "\n")
print("\nwrote", notes.relative_to(SANDBOX))
print("then run:  /compact keep the ERR-4471 repro steps and the exact failing test id; details are in NOTES.md")

**Terminal checklist** (interactive only; run `claude` in the sandbox directory):

- [ ] `/context`: note the token use and the **Memory files** list. Is `packages/api/CLAUDE.md` there? (It should not be yet.)
- [ ] Ask Claude to read `packages/api/handlers.py`, then run `/context` again. The subdirectory file is now listed.
- [ ] Mention `ERR-4471` in chat, have a few exchanges, then run `/compact keep error codes`. Ask "what was the error code?"
- [ ] Run `/compact` with no focus, and compare what Claude remembers.

> **Try it:** the stronger fix for "discovery output floods the context" is prevention, not cleanup. Rewrite one of your own noisy skills with `context: fork` and compare `/context` before and after.

## 5.8 `/memory` and auto memory

`/memory` lists every memory file (user, project and local CLAUDE.md, plus auto memory), opens the one you pick in your editor (creating it if needed), and toggles auto memory. Auto memory is Claude's own machine-local notes in `~/.claude/projects/<project>/memory/`. The first 200 lines or 25 KB of `MEMORY.md`, whichever comes first, load every session. [Theory](README.md#58-memory-and-auto-memory)

In [ ]:
def project_dir_name(path: Path) -> str:
    """Session transcripts live in ~/.claude/projects/<this>/ (non-alphanumerics -> '-')."""
    return re.sub(r"[^A-Za-z0-9]", "-", str(path))


def loaded_part_of_memory_index(text: str, max_lines: int = 200, max_bytes: int = 25_000) -> str:
    kept, size = [], 0
    for line in text.splitlines(keepends=True)[:max_lines]:
        size += len(line.encode())
        if size > max_bytes:
            break
        kept.append(line)
    return "".join(kept)


print("transcripts + auto memory for this sandbox would live under:")
print("  ~/.claude/projects/" + project_dir_name(PROJECT) + "/")

memory_index = "".join(f"- note {i}: see topic_{i}.md\n" for i in range(1, 301))
loaded = loaded_part_of_memory_index(memory_index)
print(f"\nMEMORY.md has {memory_index.count(chr(10))} lines; {loaded.count(chr(10))} load at session start")

> **Observe:** lines 201–300 of the index are invisible at session start. That is why Claude keeps `MEMORY.md` as a short index and moves detail into topic files it reads on demand. **Team knowledge belongs in the project CLAUDE.md through a reviewed PR**; auto memory is personal and unreviewed.

**Terminal checklist:**

- [ ] `/memory`: find the user, project and local entries. Note that it offers to *create* files that do not exist yet.
- [ ] Toggle auto memory off and on (this writes `autoMemoryEnabled` to `~/.claude/settings.json`).
- [ ] Say "remember: this project uses pnpm, not npm", then open the auto memory folder from `/memory` and find the note.
- [ ] In a scratch repo, run `/init` and read the CLAUDE.md it proposes. Keep only what Claude could not discover by itself.

## 5.9 Claude Code in CI/CD: headless mode

`claude -p` runs non-interactively and exits (0 on success). In CI, add `--output-format json` (or `stream-json`), `--json-schema` for validated data in `structured_output`, an exact `--allowedTools` list, `--max-turns` and `--max-budget-usd` caps, and `--bare` for runs that must not pick up local hooks, MCP servers or CLAUDE.md. [Theory](README.md#59-claude-code-in-cicd-headless-mode)

Build the argv as a **list**. That avoids shell quoting bugs, and the prompt goes right after `-p`, because several flags (`--allowedTools`, `--tools`, `--add-dir`) accept multiple values and would swallow a trailing positional prompt.

In [ ]:
def build_headless_argv(prompt: str, *, output_format: str = "json", schema: dict | None = None,
                        allowed_tools: list[str] | None = None, max_turns: int = 8,
                        max_budget_usd: float | None = None, model: str = MODEL, bare: bool = False,
                        append_system_prompt_file: str | None = None, permission_mode: str | None = None,
                        persist_session: bool = True) -> list[str]:
    argv = ["claude", "-p", prompt, "--output-format", output_format, "--model", model,
            "--max-turns", str(max_turns)]
    if output_format == "stream-json":
        argv.append("--verbose")
    if schema is not None:
        argv += ["--json-schema", json.dumps(schema)]
    if allowed_tools:
        argv += ["--allowedTools", ",".join(allowed_tools)]
    if max_budget_usd is not None:
        argv += ["--max-budget-usd", f"{max_budget_usd:.2f}"]
    if permission_mode:
        argv += ["--permission-mode", permission_mode]
    if bare:
        argv.append("--bare")
    if append_system_prompt_file:
        argv += ["--append-system-prompt-file", append_system_prompt_file]
    if not persist_session:
        argv.append("--no-session-persistence")
    return argv


argv = build_headless_argv("Summarize this project in two sentences.", allowed_tools=["Read", "Grep", "Glob"],
                           max_turns=4, max_budget_usd=0.25, bare=True,
                           append_system_prompt_file="CLAUDE.md", permission_mode="dontAsk")
print(shlex.join(argv))

### Reading the result: `json` and `stream-json`

`--output-format json` prints **one** object at the end. `stream-json` prints one JSON event per line (normally `system`/`init` first, then `assistant`/`user` messages, and a final `result`). A run that hits `--max-turns` still returns a result object, with `is_error: true` and `subtype: "error_max_turns"`, so branch on those fields, not only on the exit code.

In [ ]:
def interpret(out: dict) -> str:
    if out.get("is_error") or out.get("subtype") != "success":
        return f"FAILED ({out.get('subtype')}) after {out.get('num_turns')} turns, ${out.get('total_cost_usd', 0):.4f}"
    return f"ok in {out.get('num_turns')} turns, ${out.get('total_cost_usd', 0):.4f}, session {out.get('session_id')}"


samples = [mock_result("Acme-shop is a pricing service."),
           mock_result("", subtype="error_max_turns", is_error=True, num_turns=4)]
for s in samples:
    print(interpret(s))

stream_lines = [
    json.dumps({"type": "system", "subtype": "init", "session_id": "s-123", "model": MODEL, "tools": ["Read", "Grep"]}),
    json.dumps({"type": "assistant", "session_id": "s-123",
                "message": {"content": [{"type": "tool_use", "name": "Read", "input": {"file_path": "src/pricing.py"}}]}}),
    json.dumps({"type": "assistant", "session_id": "s-123",
                "message": {"content": [{"type": "text", "text": "apply_discount only knows SAVE10."}]}}),
    json.dumps(mock_result("apply_discount only knows SAVE10.", session_id="s-123")),
]


def parse_stream(lines: list[str]) -> dict:
    events = [json.loads(line) for line in lines if line.strip()]
    tools = [b["name"] for e in events if e["type"] == "assistant"
             for b in e["message"]["content"] if b.get("type") == "tool_use"]
    init = next(e for e in events if e["type"] == "system" and e.get("subtype") == "init")
    final = next(e for e in reversed(events) if e["type"] == "result")
    return {"session_id": init["session_id"], "tools_used": tools, "result": final["result"]}


print(parse_stream(stream_lines))

In [ ]:
# Live: one real JSON result, and a --bare variant when an API key is available
out = run_claude(build_headless_argv("Summarize this project in one sentence.", allowed_tools=["Read", "Glob"],
                                     max_turns=4, max_budget_usd=float(MAX_BUDGET_USD), persist_session=False),
                 cwd=PROJECT, mock=mock_result("acme-shop is a small pricing service with a discount helper."))
print(interpret(out))
print("top-level keys:", sorted(out))

if os.getenv("ANTHROPIC_API_KEY"):
    bare = run_claude(build_headless_argv("Which build command does this project use? Answer with the command only.",
                                          max_turns=2, max_budget_usd=float(MAX_BUDGET_USD), bare=True,
                                          append_system_prompt_file="CLAUDE.md", persist_session=False),
                      cwd=PROJECT, mock=mock_result("make build"))
    print("--bare + --append-system-prompt-file CLAUDE.md ->", bare.get("result"))

> **Observe:** with `--bare`, CLAUDE.md is **not** auto-loaded, so the project's build command reaches Claude only because we passed the file with `--append-system-prompt-file`. Use that pattern in CI: reproducible on every runner, and nothing from a teammate's `~/.claude` leaks in.

### GitHub Actions

Two options: the packaged `anthropics/claude-code-action@v1` (CLI flags go into `claude_args`), or your own script that calls `claude -p`. Both read the key from a repo secret.

In [ ]:
ACTION_WORKFLOW = """
name: Claude PR review
on:
  pull_request:
    types: [opened, synchronize, ready_for_review, reopened]
jobs:
  review:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    permissions:
      contents: read
      pull-requests: read
      issues: read
      id-token: write
    steps:
      - uses: actions/checkout@v6
        with:
          fetch-depth: 1
      - uses: anthropics/claude-code-action@v1
        with:
          anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
          prompt: |
            Review PR #${{ github.event.pull_request.number }} for correctness bugs only.
            Post each finding as an inline comment with severity and a suggested fix.
          claude_args: |
            --model claude-sonnet-5-5
            --max-turns 8
            --allowedTools "Read,Grep,Glob,Bash(gh pr diff *),mcp__github_inline_comment__create_inline_comment"
"""

wf = yaml.safe_load(ACTION_WORKFLOW)
# PyYAML follows YAML 1.1, where a bare `on` key parses as the boolean True. GitHub reads it as "on".
triggers = wf.get("on", wf.get(True))
step = wf["jobs"]["review"]["steps"][1]
print("triggers:", triggers)
print("uses:", step["uses"], "| inputs:", sorted(step["with"]))
print("claude_args:", shlex.split(step["with"]["claude_args"]))

> **Observe:** with a `prompt` input the action runs in **automation mode** on the trigger; without one it waits for `@claude` mentions. Keep CLI flags in `claude_args`. Old `@beta` inputs such as `max_turns` and `direct_prompt` no longer exist. `timeout-minutes` and `--max-turns` together cap both wall-clock time and agent loops.

> **Try it:** swap the `anthropic_api_key` line for `claude_code_oauth_token: ${{ secrets.CLAUDE_CODE_OAUTH_TOKEN }}` (the subscription route), and replace `--model claude-sonnet-5-5` with a cheaper model for a nightly, non-blocking audit.

## 5.10 Sessions: resume, name, fork

Every run is a session with a `session_id`. `--continue` picks up the latest conversation in this directory, `--resume <id|name>` a specific one, and `--fork-session` (with `--resume` or `--continue`) branches into a **new** session ID and leaves the original untouched. Sessions created with `-p` or the SDK do not appear in the interactive picker or `claude --continue`, so capture the ID. [Theory](README.md#510-sessions-resume-name-fork)

In [ ]:
def resume_argv(session_id: str, prompt: str, fork: bool = False, **kw) -> list[str]:
    argv = build_headless_argv(prompt, **kw)
    argv += ["--resume", session_id] + (["--fork-session"] if fork else [])
    return argv


base = run_claude(build_headless_argv("Remember the codeword PINEAPPLE-42 and reply only OK.", max_turns=1,
                                      max_budget_usd=float(MAX_BUDGET_USD)),
                  cwd=PROJECT, mock=mock_result("OK", session_id="11111111-aaaa-4aaa-8aaa-000000000001"))
sid = base.get("session_id")

resumed = run_claude(resume_argv(sid, "What is the codeword?", max_turns=1, max_budget_usd=float(MAX_BUDGET_USD)),
                     cwd=PROJECT, mock=mock_result("PINEAPPLE-42", session_id=sid))
forked = run_claude(resume_argv(sid, "What is the codeword? Then forget it.", fork=True, max_turns=1,
                                max_budget_usd=float(MAX_BUDGET_USD)),
                    cwd=PROJECT, mock=mock_result("PINEAPPLE-42 (forgotten in this branch)",
                                                  session_id="22222222-bbbb-4bbb-8bbb-000000000002"))

print(f"base    {sid}")
print(f"resumed {resumed.get('session_id')}  same id: {resumed.get('session_id') == sid}  -> {resumed.get('result')}")
print(f"forked  {forked.get('session_id')}  new id:  {forked.get('session_id') != sid}  -> {forked.get('result')}")

> **Observe:** resuming keeps the session ID and the history. The fork shares the history up to the fork point but gets a new ID, so the original never hears "forget it". A fork copies the **conversation, not the filesystem**: if two forks will both edit code, give each its own git worktree (`claude -w <name>`).

The same in the Agent SDK ([Chapter 3](../03-agent-sdk/README.md)), shown here but not run, because it would call your account:

```python
from claude_agent_sdk import ClaudeAgentOptions, query

async for msg in query(prompt="Design OAuth2 + PKCE instead.",
                       options=ClaudeAgentOptions(resume=sid, fork_session=True, max_turns=3)):
    ...
```

> **Try it:** start an interactive session with `claude -n auth-map`, quit, and resume it with `claude --resume auth-map`. Inside a session, `/branch try-oauth` does what `--fork-session` does. When files changed a lot since the session, a **fresh session seeded with a short summary** is often more reliable than resuming one full of stale tool output.

## 5.11 Iterative refinement with Claude Code

Claude stops when the work *looks* done, so give it a pass/fail signal. Four techniques: **2–3 concrete input/output examples** when prose is misread, **tests first** and then feed back the *specific* failures, the **interview pattern** before building in an unfamiliar domain, and **one message for interacting issues** but one at a time for independent ones. [Theory](README.md#511-iterative-refinement-with-claude-code)

The offline cells below play both roles: the "attempts" stand in for Claude's successive edits, and the helpers build the feedback you would send.

In [ ]:
# The prose spec "normalize customer names" has (at least) two reasonable readings.
def normalize_v1(name):          # reading 1: trim + lowercase
    return name.strip().lower()


def normalize_v2(name):          # reading 2: title case, no trimming
    return name.title()


# Three concrete examples remove the ambiguity, including the edge case nobody wrote down.
EXAMPLES = [
    ("  ada LOVELACE ", "Ada Lovelace"),
    ("o'brien", "O'Brien"),
    (None, None),                # a NULL from the database must pass through, not crash
]


def run_examples(fn, examples=EXAMPLES) -> list[dict]:
    """Run fn on every example; return the failures with input, expected and actual (or the exception)."""
    failures = []
    for given, expected in examples:
        try:
            actual = fn(given)
        except Exception as exc:  # noqa: BLE001 - we want the error text as feedback
            actual = f"{type(exc).__name__}: {exc}"
        if actual != expected:
            failures.append({"input": given, "expected": expected, "actual": actual})
    return failures


for fn in (normalize_v1, normalize_v2):
    print(f"{fn.__name__}: {len(run_examples(fn))} of {len(EXAMPLES)} examples fail")

> **Observe:** both implementations are faithful to the prose and both are wrong. The examples are what make "correct" checkable, and they turn directly into test cases.

### Test-driven iteration: feed back the specific failures

Write the tests first, let Claude implement, and paste back exactly what failed: the input, the expected value and the actual value or exception. Iterate until the suite is green.

In [ ]:
def attempt_1(name):             # Claude's first try: right for strings, crashes on None
    return name.strip().title()


def attempt_2(name):             # after seeing the failing None case
    return None if name is None else name.strip().title()


def feedback_message(failures: list[dict]) -> str:
    lines = ["These tests still fail. Fix the implementation, not the tests:"]
    lines += [f"- input {f['input']!r}: expected {f['expected']!r}, got {f['actual']!r}" for f in failures]
    return "\n".join(lines)


iteration_log = []
for round_no, attempt in enumerate([attempt_1, attempt_2], 1):
    failures = run_examples(attempt)
    iteration_log.append(len(failures))
    print(f"round {round_no}: {len(failures)} failing")
    if not failures:
        print("all examples pass -> stop iterating")
        break
    print(feedback_message(failures), "\n")

> **Observe:** "it still doesn't handle nulls" would leave Claude guessing. `input None: expected None, got "AttributeError: ..."` tells it exactly what to change.

### One message or one at a time?

Issues **interact** when fixing one changes the right fix for another. Send those together in one detailed message. Send **independent** issues one at a time, and verify each fix. The helper below groups issues by their `interacts_with` links. A link in either direction counts, and links chain.

In [ ]:
ISSUES = [
    {"id": "tz-dropped",   "where": "importer/parse.py",  "interacts_with": ["dedupe-by-ts"]},
    {"id": "dedupe-by-ts", "where": "importer/dedupe.py", "interacts_with": []},
    {"id": "help-typo",    "where": "cli.py",             "interacts_with": []},
    {"id": "slow-report",  "where": "reports/sql.py",     "interacts_with": []},
]


def group_issues(issues: list[dict]) -> list[list[str]]:
    """Connected components over interacts_with (undirected), in order of first appearance."""
    parent = {i["id"]: i["id"] for i in issues}

    def find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for issue in issues:
        for other in issue["interacts_with"]:
            parent[find(issue["id"])] = find(other)
    groups: dict[str, list[str]] = {}
    for issue in issues:
        groups.setdefault(find(issue["id"]), []).append(issue["id"])
    return list(groups.values())


issue_groups = group_issues(ISSUES)
for group in issue_groups:
    how = "ONE detailed message (interacting)" if len(group) > 1 else "its own message (independent)"
    print(f"{' + '.join(group):28} -> {how}")

### The interview pattern

In an unfamiliar domain, have Claude ask the questions first. The docs suggest a prompt like the one below, then a **fresh session** that implements `SPEC.md` with clean context. This is a terminal exercise, because `AskUserQuestion` needs a human to answer (in `-p` runs with `--permission-prompts none` that tool is removed).

```text
I want to add a read-through cache in front of the pricing service. Interview me in detail
using the AskUserQuestion tool. Ask about technical implementation, cache invalidation,
failure modes, edge cases and trade-offs. Don't ask obvious questions; dig into the hard
parts I might not have considered. Keep interviewing until we've covered everything,
then write a complete spec to SPEC.md.
```

> **Try it:** run that prompt in a scratch repo with `claude`. Note which questions you would not have thought of (stale reads after a price change? cache stampede when the service restarts?).

Live cell: hand Claude Code the examples as a test file and let it iterate until the tests pass. This edits only the sandbox.

In [ ]:
write(PROJECT / "src" / "names.py", """
    def normalize_name(name):
        return name.strip().title()
    """)
write(PROJECT / "tests" / "test_names.py", """
    import sys
    sys.path.insert(0, ".")
    from src.names import normalize_name

    CASES = [("  ada LOVELACE ", "Ada Lovelace"), ("o'brien", "O'Brien"), (None, None)]

    if __name__ == "__main__":
        bad = []
        for given, want in CASES:
            try:
                got = normalize_name(given)
            except Exception as exc:
                got = repr(exc)
            if got != want:
                bad.append((given, want, got))
        print("FAILED:" if bad else "OK: all cases pass", bad or "")
        sys.exit(1 if bad else 0)
    """)

examples_text = "\n".join(f"  {given!r:>18} -> {want!r}" for given, want in EXAMPLES)
refine_prompt = ("Rewrite normalize_name() in src/names.py so these examples hold:\n" + examples_text +
                 "\nThen run `python tests/test_names.py` and fix any failures. Do not edit the tests.")
out = run_claude(build_headless_argv(refine_prompt, allowed_tools=["Read", "Edit", "Bash(python tests/test_names.py)"],
                                     max_turns=6, max_budget_usd=float(MAX_BUDGET_USD), persist_session=False),
                 cwd=PROJECT, mock=mock_result("Updated normalize_name to return None for None input; all 3 cases pass."))
print(interpret(out), "\n", out.get("result"))

## 5.12 Getting started and the core loop

Claude Code is an agent, not a chat window: it **gathers context, takes action and verifies results** in a loop, using tools in your project. Permissions decide what runs without asking, and checkpoints let you rewind file edits. It runs in the terminal, VS Code, JetBrains, the Desktop app and on the web. `/init` initializes a repo with a starter CLAUDE.md. The recommended workflow is **explore → plan → code → commit**, and `--output-format stream-json` streams a headless run as it happens. [Theory](README.md#512-getting-started-and-the-core-loop)

In [ ]:
INSTALL = {
    ("terminal", "macos"):   "curl -fsSL https://claude.ai/install.sh | bash      (or: brew install --cask claude-code)",
    ("terminal", "linux"):   "curl -fsSL https://claude.ai/install.sh | bash",
    ("terminal", "wsl"):     "curl -fsSL https://claude.ai/install.sh | bash",
    ("terminal", "windows"): "irm https://claude.ai/install.ps1 | iex   in PowerShell (or: winget install Anthropic.ClaudeCode)",
    ("vscode", "*"):         "install the 'Claude Code' extension (VS Code Marketplace; Open VSX for other forks)",
    ("jetbrains", "*"):      "install the 'Claude Code' plugin from JetBrains Marketplace, plus the CLI",
    ("desktop", "*"):        "install Claude Desktop, sign in, open the Code tab",
    ("web", "*"):            "open https://claude.ai/code (a cloud session)",
}


def install_hint(surface: str, os_name: str = "*") -> str:
    return INSTALL.get((surface, os_name)) or INSTALL.get((surface, "*")) or f"unknown surface {surface!r}"


for surface, os_name in [("terminal", "macos"), ("terminal", "windows"), ("vscode", "macos"),
                         ("jetbrains", "linux"), ("desktop", "windows"), ("web", "*")]:
    print(f"{surface:9} {os_name:8} -> {install_hint(surface, os_name)}")

if RUN_CLAUDE and CLAUDE_BIN:
    print("\ninstalled:", subprocess.run([CLAUDE_BIN, "--version"], capture_output=True, text=True).stdout.strip())

> **Observe:** native installs auto-update; Homebrew and WinGet installs don't (`brew upgrade claude-code`, `winget upgrade Anthropic.ClaudeCode`). All surfaces read the same CLAUDE.md, rules, skills and settings.

**Terminal checklist: first session and `/init`** (run these yourself, in a repo you own)

1. `cd your-project && claude`. Ask "what does this project do?" and "where is the main entry point?". Claude reads files on its own.
2. Run `/init`. Claude writes a starter `CLAUDE.md` with build and test commands and conventions. If one exists, it suggests improvements instead of overwriting it.
3. Review the file: cut anything Claude can derive from the code, keep it under ~200 lines, then commit it (5.1).
4. Press `Shift+Tab` a few times and watch the status bar change mode (next cell).

### Permission modes: approval, auto-accept, plan, auto

The model below follows the documented `Shift+Tab` cycle: `default` (Manual) → `acceptEdits` → `plan`, with optional modes slotted in after `plan` (`bypassPermissions` first, `auto` last).

In [ ]:
MODES = {
    "default":           "Manual ('approval'): reads only; asks before edits and commands",
    "acceptEdits":       "Accept edits ('auto-accept'): reads, file edits, mkdir/mv/cp without asking",
    "plan":              "Plan: reads and exploration; no source edits until you approve the plan",
    "bypassPermissions": "Bypass: everything; isolated containers and VMs only",
    "auto":              "Auto: most actions; a classifier blocks risky ones",
}


def mode_cycle(auto_available: bool = True, bypass_enabled: bool = False) -> list[str]:
    return (["default", "acceptEdits", "plan"] + (["bypassPermissions"] if bypass_enabled else [])
            + (["auto"] if auto_available else []))


def shift_tab(current: str, **kw) -> str:
    cycle = mode_cycle(**kw)
    return cycle[(cycle.index(current) + 1) % len(cycle)]


mode, presses = "auto", ["auto"]          # v2.1.283+: interactive sessions start in auto mode
for _ in range(4):
    mode = shift_tab(mode)
    presses.append(mode)
print(" -> ".join(presses))
for name in presses[1:4]:
    print(f"  {name:12} {MODES[name]}")

### Explore → plan → code → commit

The docs' recommended workflow, written as data so we can check it. Explore and plan run in **plan mode** (nothing is edited); code runs after the plan is approved and includes a verification step; commit comes last.

In [ ]:
GOOD_WORKFLOW = [
    ("explore", "plan", "read src/auth and explain how sessions and login work"),
    ("plan",    "plan", "I want to add Google OAuth. Which files change? What's the session flow? Create a plan."),
    ("code",    "acceptEdits", "implement the OAuth flow from your plan. write tests for the callback handler, "
                               "run the test suite and fix any failures"),
    ("commit",  "acceptEdits", "commit with a descriptive message and open a PR"),
]
RUSHED_WORKFLOW = [
    ("code",    "auto", "add Google OAuth"),
    ("commit",  "auto", "commit it"),
]


def workflow_problems(steps: list[tuple[str, str, str]]) -> list[str]:
    phases = [p for p, _, _ in steps]
    problems = []
    if phases[:2] != ["explore", "plan"]:
        problems.append("start by exploring and planning before any code changes")
    problems += [f"{p} should run in plan mode" for p, mode, _ in steps if p in ("explore", "plan") and mode != "plan"]
    code_prompts = [prompt for p, _, prompt in steps if p == "code"]
    if not any("test" in prompt for prompt in code_prompts):
        problems.append("the code step needs a check Claude can run (tests)")
    if phases[-1:] != ["commit"]:
        problems.append("finish with a commit/PR step")
    return problems


for name, wf in [("good", GOOD_WORKFLOW), ("rushed", RUSHED_WORKFLOW)]:
    print(f"{name:7} -> {workflow_problems(wf) or 'OK'}")

> **Observe:** the rushed version is fine for a one-sentence change (a typo, a log line). For an unfamiliar, multi-file feature it skips exactly the steps that prevent rework (5.6).

### Streaming mode: `--output-format stream-json`

`json` prints one object at the end. `stream-json` prints one JSON event per line **while the run is in progress**: a `system`/`init` event first, then `assistant`/`user` messages, and the `result` last. With `--include-partial-messages` you also get `stream_event` lines that wrap raw API events. Text arrives as `content_block_delta` events with a `text_delta`.

In [ ]:
def stream_argv(prompt: str) -> list[str]:
    return ["claude", "-p", prompt, "--output-format", "stream-json", "--verbose", "--include-partial-messages",
            "--model", MODEL, "--max-turns", "1", "--max-budget-usd", MAX_BUDGET_USD, "--no-session-persistence"]


def _delta(text: str) -> str:
    return json.dumps({"type": "stream_event", "session_id": "s-9", "uuid": f"u-{text[:4]}", "parent_tool_use_id": None,
                       "event": {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": text}}})


FINAL = "Recursion is when a function calls itself on a smaller input until it hits a base case."
MOCK_STREAM = [
    json.dumps({"type": "system", "subtype": "init", "session_id": "s-9", "model": MODEL, "tools": ["Read", "Grep"]}),
    json.dumps({"type": "stream_event", "session_id": "s-9", "uuid": "u-start", "parent_tool_use_id": None,
                "event": {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}}}),
    *[_delta(chunk) for chunk in ["Recursion is when a function ", "calls itself on a smaller input ",
                                  "until it hits a base case."]],
    json.dumps({"type": "stream_event", "session_id": "s-9", "uuid": "u-stop", "parent_tool_use_id": None,
                "event": {"type": "content_block_stop", "index": 0}}),
    json.dumps({"type": "assistant", "session_id": "s-9", "message": {"content": [{"type": "text", "text": FINAL}]}}),
    json.dumps(mock_result(FINAL, session_id="s-9", num_turns=1)),
]


def stream_lines(argv: list[str], cwd: Path, mock_lines: list[str]):
    """Yield stdout lines as they arrive (live), or the mock lines (offline)."""
    print(f"$ cd {cwd.name} && {shlex.join(argv)[:300]}")
    if not (RUN_CLAUDE and CLAUDE_BIN):
        print("  [offline] not executed; replaying a mock stream\n")
        yield from mock_lines
        return
    with subprocess.Popen(argv, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
                          env=os.environ.copy()) as proc:
        yield from proc.stdout


def follow_stream(lines) -> dict:
    """Print text deltas as they arrive (like the jq filter in the README) and return a summary."""
    session_id, chunks, final = None, [], None
    for line in lines:
        if not line.strip():
            continue
        ev = json.loads(line)
        if ev.get("type") == "system" and ev.get("subtype") == "init":
            session_id = ev.get("session_id")
        elif ev.get("type") == "stream_event":
            inner = ev.get("event", {})
            if inner.get("type") == "content_block_delta" and inner.get("delta", {}).get("type") == "text_delta":
                chunks.append(inner["delta"]["text"])
                print(inner["delta"]["text"], end="", flush=True)
        elif ev.get("type") == "result":
            final = ev
    print()
    return {"session_id": session_id, "streamed_text": "".join(chunks), "final": final}


streamed = follow_stream(stream_lines(stream_argv("Explain recursion in one sentence."), PROJECT, MOCK_STREAM))
print("session:", streamed["session_id"], "| streamed == final result:",
      streamed["streamed_text"] == (streamed["final"] or {}).get("result"))
print(interpret(streamed["final"] or {}))

> **Observe:** the deltas reconstruct the same text that the final `result` line carries. A dashboard shows the deltas live; a CI gate should still decide on the final `result` (`is_error`, `subtype`), exactly as in 5.9.

> **Try it:** add a line with `"type": "assistant"` that contains a `tool_use` block and a non-null `parent_tool_use_id`, then extend `follow_stream` to print subagent tool calls separately from main-session text.

### Claude Code or Computer Use?

The Academy "deploy Anthropic apps" outcome pairs **Claude Code** (development automation) with **Computer Use** (GUI automation). Computer Use comes in two forms: the **API toolset** `computer_toolset_20260801`, which your own app hosts in a sandbox, and **computer use inside Claude Code**, an interactive research preview that drives your real desktop. The cells below are offline: they build the API request (Chapter 2 runs the full loop), check whether Claude Code's computer use is even available for a setup, follow Claude Code's tool order, and audit a deployment against the documented safety precautions.

In [ ]:
def computer_use_request(task: str, model: str = MODEL) -> dict:
    """The Messages API request a hosted Computer Use app sends first (offline: built, not sent)."""
    return {
        "model": model,
        "max_tokens": 1024,
        "tools": [{"type": "computer_toolset_20260801"}],   # one entry, no name: 17 member tools
        "messages": [{"role": "user", "content": task}],
    }


def claude_code_computer_use_blockers(*, os_name: str, surface: str, plan: str, auth: str,
                                      headless: bool) -> list[str]:
    """Why Claude Code's own computer use is unavailable for this setup ([] = available)."""
    blockers = []
    if headless:
        blockers.append("needs an interactive session: not available with -p, so not in CI")
    if plan not in {"pro", "max"}:
        blockers.append("research preview for Pro and Max plans only (not Team or Enterprise)")
    if auth != "claude.ai":
        blockers.append("needs a claude.ai login: not available through Bedrock, Google Cloud or Foundry")
    if surface == "cli" and os_name != "macos":
        blockers.append("the CLI version is macOS only (the Desktop app adds Windows)")
    if surface == "desktop" and os_name not in {"macos", "windows"}:
        blockers.append("the Desktop app version is macOS and Windows only")
    return blockers


print(json.dumps(computer_use_request("Open the claims app and export today's report as PDF."), indent=2))
print()
SETUPS = {
    "dev on a Mac, Max plan, CLI":        dict(os_name="macos", surface="cli", plan="max", auth="claude.ai", headless=False),
    "nightly cron on Google Cloud":       dict(os_name="linux", surface="cli", plan="enterprise", auth="google-cloud", headless=True),
    "Windows laptop, Pro plan, CLI":      dict(os_name="windows", surface="cli", plan="pro", auth="claude.ai", headless=False),
    "Windows laptop, Pro plan, Desktop":  dict(os_name="windows", surface="desktop", plan="pro", auth="claude.ai", headless=False),
}
for name, setup in SETUPS.items():
    blockers = claude_code_computer_use_blockers(**setup)
    print(f"{name:36} -> {'available (enable computer-use in /mcp or Desktop settings)' if not blockers else 'blocked'}")
    for b in blockers:
        print(f"{'':39}- {b}")

> **Observe:** the nightly job hits all four blockers. Unattended GUI automation, or any setup that authenticates through a cloud provider, is built on the API toolset instead, where your app runs the actions on a virtual display.

Inside Claude Code, computer use is the **last** resort. Claude tries the most precise tool first, and the same order is good system design:

In [ ]:
CLAUDE_CODE_TOOL_ORDER = [
    ("MCP server",       lambda t: t.get("mcp_server", False)),
    ("Bash",             lambda t: t.get("shell_command", False)),
    ("Claude in Chrome", lambda t: t.get("browser", False) and t.get("chrome_extension", False)),
    ("computer use",     lambda t: True),          # native apps, simulators, tools with no API
]


def claude_code_tool_for(task: dict) -> str:
    return next(name for name, applies in CLAUDE_CODE_TOOL_ORDER if applies(task))


TASKS = {
    "create a Jira ticket (Jira MCP server installed)": {"mcp_server": True},
    "run the test suite":                               {"shell_command": True},
    "check the signup page in the browser":             {"browser": True, "chrome_extension": True},
    "click through a SwiftUI menu bar app":             {},
}
for task, traits in TASKS.items():
    print(f"{task:50} -> {claude_code_tool_for(traits)}")

# Documented precautions for a hosted Computer Use deployment (computer use tool docs)
PRECAUTIONS = {
    "isolated_vm":        "run in a dedicated VM or container with minimal privileges",
    "domain_allowlist":   "limit internet access to an allowlist of domains",
    "no_credentials":     "keep login credentials and sensitive data out of the model's reach",
    "human_confirmation": "ask a human to confirm actions with real-world consequences",
}


def deployment_gaps(config: dict) -> list[str]:
    return [advice for key, advice in PRECAUTIONS.items() if not config.get(key)]


naive = {"isolated_vm": False, "domain_allowlist": False, "no_credentials": False, "human_confirmation": False}
hardened = {k: True for k in PRECAUTIONS}
print("\nnaive deployment gaps:", *deployment_gaps(naive), sep="\n  - ")
print("hardened deployment gaps:", deployment_gaps(hardened) or "none")

> **Observe:** screen control is the broadest and slowest option, so it comes last both in Claude Code and in your designs. A hosted deployment that skips the precautions turns every web page and document Claude sees into a possible prompt-injection channel with real power behind it.

> **Try it:** start from Chapter 2's practice 2.7 harness and add a `cache_control` breakpoint to `computer_use_request()`'s tool entry, then write a helper that keeps only the last three screenshot results in a long message history (screenshots cost roughly 1,000–1,800 input tokens each).

## Mini-project: a CI review bot with structured output

Combine the chapter into one script-shaped pipeline, the kind you would run from the workflow above:

1. build a unified diff (a PR that introduces two bugs)
2. run a **fresh, independent** headless review: diff on stdin, `--json-schema`, read-only tools, turn and budget caps, and `--bare` + `--append-system-prompt-file CLAUDE.md` when an API key is available
3. turn `structured_output` into a GitHub "create review" payload (dry run, nothing is posted)
4. gate the job: fail on critical/high findings, and treat an incomplete review as a separate status
5. re-review after a fix commit, passing the previous findings so only new or unresolved issues come back

In [ ]:
OLD = (PROJECT / "src" / "pricing.py").read_text()
NEW = """DISCOUNTS = {"SAVE10": 0.10, "SAVE50": 0.50}

def apply_discount(price: float, code: str) -> float:
    rate = DISCOUNTS.get(code.upper())
    return round(price * rate, 2)
"""


def unified_diff(old: str, new: str, path: str) -> str:
    return "".join(difflib.unified_diff(old.splitlines(keepends=True), new.splitlines(keepends=True),
                                        fromfile=f"a/{path}", tofile=f"b/{path}"))


DIFF = unified_diff(OLD, NEW, "src/pricing.py")
print(DIFF)

REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "findings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "line": {"type": "integer"},
                    "severity": {"enum": ["critical", "high", "medium", "low"]},
                    "issue": {"type": "string"},
                    "suggested_fix": {"type": "string"},
                },
                "required": ["file", "line", "severity", "issue", "suggested_fix"],
            },
        }
    },
    "required": ["findings"],
}

In [ ]:
MOCK_FINDINGS = [
    {"file": "src/pricing.py", "line": 5, "severity": "critical",
     "issue": "Multiplies by the discount rate instead of (1 - rate): SAVE10 now charges 10% of the price.",
     "suggested_fix": "return round(price * (1 - rate), 2)"},
    {"file": "src/pricing.py", "line": 4, "severity": "high",
     "issue": "Unknown codes make rate None, so the multiplication raises TypeError.",
     "suggested_fix": "rate = DISCOUNTS.get(code.upper(), 0.0)"},
]


def review_diff(diff: str, previous: list[dict] | None = None, mock_findings: list[dict] | None = None) -> dict:
    prompt = ("You are an independent code reviewer. The unified diff is on stdin. Report only concrete "
              "correctness bugs in added or changed lines, using line numbers of the new file.")
    if previous:
        prompt += ("\nThese findings were reported on an earlier commit:\n" + json.dumps(previous, indent=1) +
                   "\nReport only issues that are new or still unresolved in this diff.")
    use_bare = bool(os.getenv("ANTHROPIC_API_KEY"))
    argv = build_headless_argv(prompt, schema=REVIEW_SCHEMA, allowed_tools=["Read", "Grep", "Glob"],
                               max_turns=6, max_budget_usd=0.50, bare=use_bare,
                               append_system_prompt_file="CLAUDE.md" if use_bare else None,
                               permission_mode="dontAsk", persist_session=False)
    findings = MOCK_FINDINGS if mock_findings is None else mock_findings
    return run_claude(argv, cwd=PROJECT, stdin=diff, mock=mock_result("", structured={"findings": findings}))


SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}


def to_github_review(findings: list[dict], commit_sha: str) -> dict:
    """Body for POST /repos/{owner}/{repo}/pulls/{number}/reviews (dry run: we only print it)."""
    ordered = sorted(findings, key=lambda f: (SEVERITY_RANK[f["severity"]], f["file"], f["line"]))
    return {
        "commit_id": commit_sha,
        "event": "COMMENT",
        "body": f"Claude review: {len(ordered)} finding(s).",
        "comments": [{"path": f["file"], "line": f["line"], "side": "RIGHT",
                      "body": f"**[{f['severity']}]** {f['issue']}\n\nSuggested fix: `{f['suggested_fix']}`"}
                     for f in ordered],
    }


def gate(out: dict, fail_on: frozenset = frozenset({"critical", "high"})) -> int:
    """0 = pass, 1 = blocking findings, 2 = review incomplete (do not silently pass)."""
    if out.get("is_error") or "structured_output" not in out:
        return 2
    return 1 if any(f["severity"] in fail_on for f in out["structured_output"]["findings"]) else 0


first = review_diff(DIFF)
print(interpret(first))
review = to_github_review(first.get("structured_output", {}).get("findings", []), commit_sha="abc1234")
print(json.dumps(review, indent=2)[:1200])
print("exit code:", gate(first))

In [ ]:
# The author pushes a fix for the critical bug only. Re-review with the previous findings in context.
FIXED = NEW.replace("price * rate", "price * (1 - rate)")
DIFF2 = unified_diff(OLD, FIXED, "src/pricing.py")
previous = first.get("structured_output", {}).get("findings", [])


def issue_key(f: dict) -> tuple[str, str]:
    return f["file"], re.sub(r"\W+", " ", f["issue"].lower()).strip()[:60]


second = review_diff(DIFF2, previous=previous, mock_findings=[MOCK_FINDINGS[1]])
current = second.get("structured_output", {}).get("findings", [])
seen_before = {issue_key(f) for f in previous}
print("still unresolved:", [f["issue"] for f in current if issue_key(f) in seen_before])
print("new:             ", [f["issue"] for f in current if issue_key(f) not in seen_before] or "-")
print("exit code:", gate(second))
print("incomplete run ->", gate(mock_result("", subtype="error_max_turns", is_error=True)))

> **Observe:** the reviewer never saw the reasoning of whoever wrote the code, the output is schema-validated data instead of prose, and the gate fails *closed* (exit 2) when the run hits a limit, instead of reporting "no findings". Passing the previous findings keeps the second review from re-posting duplicates.

> **Try it:** wire it into CI. Save `build_headless_argv`, `run_claude` (without the offline branch), `review_diff`, `to_github_review` and `gate` as `scripts/claude_review.py`, then call it from a job like the one below. Use the notebook as the test bed: change `MOCK_FINDINGS` and check `gate()` and the payload before spending a cent.

In [ ]:
SCRIPT_WORKFLOW = """
name: Claude review gate
on: pull_request
jobs:
  review:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    permissions:
      contents: read
      pull-requests: write
    steps:
      - uses: actions/checkout@v6
        with:
          fetch-depth: 0
      - name: Install Claude Code
        run: |
          curl -fsSL https://claude.ai/install.sh | bash
          echo "$HOME/.local/bin" >> "$GITHUB_PATH"
      - name: Review the diff
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          git diff "origin/${{ github.base_ref }}...HEAD" > pr.diff
          python3 scripts/claude_review.py pr.diff
"""
wf = yaml.safe_load(SCRIPT_WORKFLOW)
print([s.get("name", s.get("uses")) for s in wf["jobs"]["review"]["steps"]])

## Clean up

In [ ]:
shutil.rmtree(SANDBOX, ignore_errors=True)
print("removed", SANDBOX)

## Recap

- CLAUDE.md files are **concatenated** from broad to specific; ancestors load at launch, subdirectories lazily, and `CLAUDE.local.md` is personal and gitignored.
- `@imports` organize but load at launch (four hops max, never inside backticks); `.claude/rules/` with `paths:` globs load only when a matching file is read, written or edited, and broken YAML silently makes a rule unscoped.
- Skills replaced commands (skill wins on a name clash; personal shadows project). Use `disable-model-invocation` for side effects, `context: fork` + `agent: Explore` for noisy discovery, and `disallowed-tools` (not `allowed-tools`) to restrict.
- Plan mode writes nothing until approval; `/compact` is lossy, so persist exact facts first; auto memory is personal, CLAUDE.md is team knowledge.
- CI: `claude -p` + `--output-format json` + `--json-schema` -> `structured_output`, with `--allowedTools`, `--max-turns`, `--max-budget-usd` and `--bare`. Branch on `is_error`/`subtype`, not just the exit code.
- Sessions: capture `session_id`, `--resume` to continue, `--fork-session` (or `/branch`) to diverge, and worktrees when forks edit code.
- Iteration: concrete I/O examples become tests; feed back the exact failing input, expected and actual values; send interacting issues together and independent ones one at a time; interview first in unfamiliar domains.
- Getting started: Claude Code is an agentic loop (gather context → act → verify) on many surfaces; `/init` bootstraps CLAUDE.md; `Shift+Tab` cycles Manual → Accept edits → Plan (→ Auto); explore → plan → code → commit; `stream-json --verbose` streams events, and the last line is the `result`.
- Claude Code automates development work; Computer Use automates GUIs. Host the `computer_toolset_20260801` API toolset in a sandbox for unattended jobs; Claude Code's own computer use is an interactive, Pro/Max, claude.ai-login research preview and the last tool it reaches for.

Next: [homework](02_homework.ipynb)